# 02. IVI 전처리 노트북
## 사회적 고립 취약도 지수 (Isolation Vulnerability Index)

**프로젝트**: 비가 오면 누가 고립되는가 — 서울시 고령 1인가구의 복합위기 취약지역 탐지  
**분석 단위**: 행정동  
**대상 연도**: 2019–2021 (최종 IVI는 2021년 기준)

### 사용 데이터 및 역할

| 파일 | 공간 단위 | 시간 단위 | 역할 |
|------|----------|----------|------|
| 독거노인+현황(연령별_동별) | **동별** ✅ | 연간 | elderly_alone (핵심) |
| 서울_등록인구_동별_연령별_데이터 | **동별** ✅ | 연간 | total_population + elderly_population |
| 장애인+현황(장애유형별_동별) | 구별만 ⚠️ | 연간 | ~~IVI 제외~~ (구별 수준으로 동별 산출 부적합) |

### IVI 공식 (2개 지표 동일가중 평균)
$$\text{IVI} = \frac{\text{elderly\_ratio\_score} + \text{elderly\_alone\_score}}{2}$$

| 구성 요소 | 계산식 | 데이터 수준 |
|----------|--------|------------|
| elderly_ratio_score | 동별 65세 이상 / 동별 전체 인구 | **동별 ✅** |
| elderly_alone_score | 동별 독거노인 / 동별 65세 이상 | **동별 ✅** |

---
## 1. 라이브러리 및 경로 설정

분석에 필요한 라이브러리를 불러오고, 파일 경로를 설정합니다.  
모든 경로는 프로젝트 루트 기준으로 작성하여 다른 환경에서도 동작하도록 합니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── 프로젝트 루트 ──
BASE_DIR = Path("c:/Tsum2026/T_SUM2026")

# ── 원본 데이터 폴더 ──
RAW_IVI_DIR  = BASE_DIR / "data" / "raw" / "IVI_social_isolation"

# ── 전처리 결과 저장 폴더 ──
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── 진단표 폴더 ──
DIAG_DIR = BASE_DIR / "outputs" / "tables"

# ── 파일 경로 ──
FILE_ELDERLY_ALONE = RAW_IVI_DIR / "독거노인+현황(연령별_동별)_2019_2021.csv"
FILE_POPULATION    = RAW_IVI_DIR / "서울_등록인구_동별_연령별_데이터.csv"   # 동별 연령별 버전
FILE_DISABLED      = RAW_IVI_DIR / "장애인+현황(장애유형별_동별)_20260430042931.csv"

# ── 경로 존재 확인 ──
for label, path in [
    ("IVI 원본 폴더",   RAW_IVI_DIR),
    ("전처리 저장 폴더", PROCESSED_DIR),
    ("독거노인 파일",    FILE_ELDERLY_ALONE),
    ("등록인구 파일",    FILE_POPULATION),
    ("장애인 파일",      FILE_DISABLED),
]:
    mark = "✓" if path.exists() else "✗ 없음!"
    print(f"  [{mark}] {label}: {path}")


  [✓] IVI 원본 폴더: c:\Tsum2026\T_SUM2026\data\raw\IVI_social_isolation
  [✓] 전처리 저장 폴더: c:\Tsum2026\T_SUM2026\data\processed
  [✓] 독거노인 파일: c:\Tsum2026\T_SUM2026\data\raw\IVI_social_isolation\독거노인+현황(연령별_동별)_2019_2021.csv
  [✓] 등록인구 파일: c:\Tsum2026\T_SUM2026\data\raw\IVI_social_isolation\서울_등록인구_동별_연령별_데이터.csv
  [✓] 장애인 파일: c:\Tsum2026\T_SUM2026\data\raw\IVI_social_isolation\장애인+현황(장애유형별_동별)_20260430042931.csv


---
## 2. 진단표 불러오기 및 IVI 사용 데이터 확인

`01_data_check.ipynb`에서 생성된 진단표를 불러와 IVI에 사용할 데이터셋과 각각의 역할을 확인합니다.  
진단표는 각 파일의 공간 단위, 변환 등급, 권장 전처리 방법 등을 담고 있습니다.

In [2]:
# 진단표 불러오기
df_diag = pd.read_csv(DIAG_DIR / "ivi_dataset_diagnosis.csv", encoding="utf-8-sig")

# 핵심 컬럼만 선택해서 확인
VIEW_COLS = [
    "dataset_name", "spatial_unit", "has_dong",
    "conversion_grade", "use_role", "suggested_preprocess"
]
print("=== IVI 진단표 (핵심 컬럼) ===")
pd.set_option("display.max_colwidth", 60)
df_diag[VIEW_COLS]

=== IVI 진단표 (핵심 컬럼) ===


,dataset_name,spatial_unit,has_dong,conversion_grade,use_role,suggested_preprocess
0,1인가구_연령별,구별,False,B,핵심후보,구별 고령 1인가구 수 추출 후 구 단위 보조변수로 사용
1,노인여가복지시설,구별,False,B,보조후보,구별 시설수 합산; 시설이 많을수록 취약도 낮으므로 부족도로 반전 필요
2,독거노인현황,구별+동별,True,A,핵심후보,동별 독거노인 수 추출 후 고령인구 대비 비율 산출
3,등록인구+등록외국인,구별+동별,True,A,핵심후보,동별 65세 이상 한국인 합산; 외국인 데이터는 핵심 변수 제외 검토
4,등록외국인,구별+동별,True,C,제외검토,등록외국인은 IVI 핵심 변수 제외 검토; 보조 분석 시 별도 활용
5,장애유형별_동별,구별+동별,True,A,보조후보,동별 장애인 인원 합산 후 전체 인구 대비 비율 산출


---
## 3. 독거노인현황 불러오기 및 전처리

### 파일 구조
- **헤더 행**: 4행 (행0=연도, 행1=합계, 행2=분류, 행3=계/연령)  
- **지리 컬럼**: 동별(1)=서울특별시, 동별(2)=구, 동별(3)=동  
- **데이터**: 5행째부터

### 연도별 컬럼 위치
1년치 = 소계·수급권자·저소득·일반 × 계/65~79세/80세이상 = **12열**

| 연도 | 소계/계 컬럼 인덱스 |
|------|-------------------|
| 2019 | 3 |
| 2020 | 15 (= 3 + 12) |
| 2021 | 27 (= 3 + 24) |

### 주의사항
- `'-'` 값은 0으로 처리 (서울시 Open Data 관행)
- `소계` 또는 `합계` 등 요약행은 반드시 제거 후 분석

In [3]:
# ── 공용 유틸리티 함수 ──

def read_csv_safe(filepath, encoding_list=None):
    """여러 인코딩을 순서대로 시도해 CSV를 읽는 함수."""
    if encoding_list is None:
        encoding_list = ["cp949", "utf-8-sig", "euc-kr", "utf-8"]
    for enc in encoding_list:
        try:
            df = pd.read_csv(filepath, header=None, encoding=enc)
            print(f"  → 인코딩 '{enc}' 성공 ({df.shape[0]}행 × {df.shape[1]}열)")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"파일을 읽을 수 없습니다: {filepath}")


def to_numeric_safe(series):
    """'-', 공백, NaN을 0으로 처리한 뒤 숫자로 변환."""
    return (
        pd.to_numeric(
            series.astype(str).str.strip().replace({"-": "0", "": "0", "nan": "0"}),
            errors="coerce"
        ).fillna(0)
    )


# 요약행 키워드 (구/동 컬럼에서 제거할 값)
SUMMARY_KEYWORDS = {"소계", "합계", "계", "nan", "", "서울특별시"}

In [4]:
# ── 독거노인현황 파일 구조 확인 ──
print("=== 독거노인현황 파일 ===\n읽는 중...")
df_raw_ea = read_csv_safe(FILE_ELDERLY_ALONE)

print("\n헤더 4행 (처음 10열):")
print(df_raw_ea.iloc[:4, :10].to_string(index=True))

print("\n데이터 시작 3행 (처음 10열):")
print(df_raw_ea.iloc[4:7, :10].to_string(index=True))

=== 독거노인현황 파일 ===
읽는 중...
  → 인코딩 'utf-8-sig' 성공 (458행 × 39열)

헤더 4행 (처음 10열):
       0      1      2     3       4       5              6              7              8      9
0  동별(1)  동별(2)  동별(3)  2019    2019    2019           2019           2019           2019   2019
1  동별(1)  동별(2)  동별(3)    합계      합계      합계             합계             합계             합계     합계
2  동별(1)  동별(2)  동별(3)    소계      소계      소계  국민기초생활보장 수급권자  국민기초생활보장 수급권자  국민기초생활보장 수급권자  저소득노인
3  동별(1)  동별(2)  동별(3)     계  65~79세  80세 이상              계         65~79세         80세 이상      계

데이터 시작 3행 (처음 10열):
       0    1      2       3       4      5      6      7      8      9
4  서울특별시   소계     소계  343567  255395  88172  80376  56572  23804  20131
5  서울특별시  종로구     소계    9149    6653   2496   1445   1099    346    528
6  서울특별시  종로구  청운효자동     476     313    163     53     27     26     32


In [5]:
# ── 독거노인현황 전처리 ──

# 1) 데이터 행 추출 (헤더 4행 건너뜀)
df_ea = df_raw_ea.iloc[4:].reset_index(drop=True).copy()
df_ea = df_ea.rename(columns={0: "시", 1: "gu", 2: "dong"})

# 2) 연도별 소계/계 컬럼 위치
ELDERLY_ALONE_COL = {2019: 3, 2020: 15, 2021: 27}

# 3) 연도별 long format 변환
records_ea = []
for year, col_idx in ELDERLY_ALONE_COL.items():
    tmp = df_ea[["gu", "dong", col_idx]].copy()
    tmp.columns = ["gu", "dong", "elderly_alone"]
    tmp["year"] = year
    records_ea.append(tmp)

df_elderly_all = pd.concat(records_ea, ignore_index=True)

# 4) 문자열 공백 제거
df_elderly_all["gu"]   = df_elderly_all["gu"].astype(str).str.strip()
df_elderly_all["dong"] = df_elderly_all["dong"].astype(str).str.strip()

# 5) 구 이름 정규화 ('동대문' → '동대문구')
GU_NAME_MAP = {"동대문": "동대문구"}
df_elderly_all["gu"] = df_elderly_all["gu"].replace(GU_NAME_MAP)

# 6) 동 이름 정규화 ('정능X동' → '정릉X동')
DONG_NAME_MAP = {
    "정능1동": "정릉1동", "정능2동": "정릉2동",
    "정능3동": "정릉3동", "정능4동": "정릉4동",
}
df_elderly_all["dong"] = df_elderly_all["dong"].replace(DONG_NAME_MAP)

# 7) 요약행 제거
df_elderly_all = df_elderly_all[
    (~df_elderly_all["dong"].isin(SUMMARY_KEYWORDS)) &
    (~df_elderly_all["gu"].isin(SUMMARY_KEYWORDS))
].reset_index(drop=True)

# 8) '-' 처리 및 숫자 변환
df_elderly_all["elderly_alone"] = to_numeric_safe(df_elderly_all["elderly_alone"])

# 9) dong_key 생성
df_elderly_all["dong_key"] = df_elderly_all["gu"] + "_" + df_elderly_all["dong"]

# 10) dong_key 중복 제거 (원자료 중복 입력 방지 — 강동구_강일동 등)
dup_ea = df_elderly_all[df_elderly_all.duplicated(["year", "dong_key"], keep=False)]
if len(dup_ea) > 0:
    print(f"⚠️ 독거노인 원자료 중복 {len(dup_ea)}건 → 중복 내역:")
    print(dup_ea[["year", "gu", "dong", "elderly_alone"]].to_string(index=False))
    print("  → 첫 번째 행 유지 (값이 동일한 경우)")
df_elderly_all = df_elderly_all.drop_duplicates(["year", "dong_key"], keep="first").reset_index(drop=True)

# 11) 결과 확인
print(f"\n독거노인현황 처리 완료: {df_elderly_all.shape[0]}행")
print(f"연도별 행정동 수:")
print(df_elderly_all.groupby("year")["dong_key"].nunique())
print(f"\n샘플 (2021년):")
df_elderly_all[df_elderly_all["year"] == 2021].head(5)


⚠️ 독거노인 원자료 중복 6건 → 중복 내역:
 year  gu dong  elderly_alone
 2019 강동구  강일동            627
 2019 강동구  강일동              0
 2020 강동구  강일동            870
 2020 강동구  강일동              0
 2021 강동구  강일동              0
 2021 강동구  강일동           1091
  → 첫 번째 행 유지 (값이 동일한 경우)

독거노인현황 처리 완료: 1281행
연도별 행정동 수:
year
2019    427
2020    427
2021    427
Name: dong_key, dtype: int64

샘플 (2021년):


,gu,dong,elderly_alone,year,dong_key
854,종로구,청운효자동,514,2021,종로구_청운효자동
855,종로구,사직동,490,2021,종로구_사직동
856,종로구,삼청동,189,2021,종로구_삼청동
857,종로구,부암동,418,2021,종로구_부암동
858,종로구,평창동,703,2021,종로구_평창동


---
## 4. 등록인구 불러오기 및 행정동별 전체 인구·65세 이상 인구 추출

### 파일 구조 (서울_등록인구_동별_연령별_데이터.csv)
- **`동별`**: 합계 → 구명(subtotal) → 동명 순으로 반복
- **`연령별`**: 합계, 65~69세, 70~74세, 75~79세, 80~84세, 85~89세, 90~94세, 95~99세, 100세 이상
- **`성별`**: 계(전체)
- **연간 대표 컬럼**: `2019 년`, `2020 년`, `2021 년` (연간 합산값)

### 전처리 전략
1. `gu` forward-fill: `동별` 값이 25개 자치구명일 때 현재 구 갱신 → 이후 동 행에 채워넣기
2. `합계`(서울 전체) 및 구 subtotal 행 제거
3. 연도별 `YYYY 년` 결측 행 제거 후 각각 처리
4. `연령별 == 합계` → `total_population`
5. `65~69세 ~ 100세 이상` 합산 → `elderly_population`
6. 최종 산출: **long format** `[year, gu, dong, dong_key, total_population, elderly_population]`

### ✅ 이전 대비 개선
| 항목 | 이전 (구별 파일) | 현재 (동별 파일) |
|------|----------------|----------------|
| total_population | 구별 ⚠️ | **동별 ✅** |
| elderly_population | 구별 ⚠️ | **동별 ✅** |
| elderly_ratio | 구 간 차이만 반영 | **동 간 차이 반영** ✅ |
| elderly_alone_ratio | 동/구 혼합 | **동/동 ✅** |
| 연도 | 2019–2021 | 2019–2021 (동일) |

In [6]:
# ── 등록인구(동별_연령별) 파일 구조 확인 ──
print("=== 등록인구(동별_연령별) 파일 ===\n읽는 중...")

# 파일 내 따옴표가 이스케이프되지 않아 C 파서 파싱 오류 발생
# → quoting=3 (QUOTE_NONE) 으로 읽은 뒤 따옴표를 수동으로 제거
df_raw_pop = pd.read_csv(FILE_POPULATION, encoding='utf-8-sig', quoting=3)

# 컬럼명 따옴표 제거
df_raw_pop.columns = [c.strip('"') for c in df_raw_pop.columns]

# 문자열 값 따옴표 제거
for col in df_raw_pop.select_dtypes(include='object').columns:
    df_raw_pop[col] = df_raw_pop[col].astype(str).str.strip('"').str.strip()

print(f"  → 읽기 성공: {df_raw_pop.shape[0]}행 × {df_raw_pop.shape[1]}열")
print(f"  컬럼: {list(df_raw_pop.columns)}")

# 2021 연간 컬럼 확인 (분기 아닌 연간 합산 컬럼)
col_2021_candidates = [
    c for c in df_raw_pop.columns
    if '2021' in c and '/' not in c and 'Unnamed' not in c
]
print(f"\n  2021 연간 컬럼 후보: {col_2021_candidates}")

# 동별 값 분포 (처음 30행)
COL_DONG_TMP = df_raw_pop.columns[0]
COL_AGE_TMP  = df_raw_pop.columns[1]
print(f"\n  동별 컬럼 처음 35개 unique 값: {list(df_raw_pop[COL_DONG_TMP].unique()[:35])}")
print(f"\n  연령별 컬럼 unique 값: {sorted(df_raw_pop[COL_AGE_TMP].unique())}")


=== 등록인구(동별_연령별) 파일 ===
읽는 중...
  → 읽기 성공: 4095행 × 20열
  컬럼: ['동별', '연령별', '항목', '단위', '2019. 1/4', '2019. 2/4', '2019. 3/4', '2019. 4/4', '2020. 1/4', '2020. 2/4', '2020. 3/4', '2020. 4/4', '2021. 1/4', '2021. 2/4', '2021. 3/4', '2021. 4/4', '2019 년', '2020 년', '2021 년', 'Unnamed: 19']

  2021 연간 컬럼 후보: ['2021 년']

  동별 컬럼 처음 35개 unique 값: ['합계', '종로구', '청운효자동', '사직동', '삼청동', '부암동', '평창동', '무악동', '교남동', '가회동', '종로1.2.3.4가동', '종로5.6가동', '이화동', '혜화동', '창신1동', '창신2동', '창신3동', '숭인1동', '숭인2동', '중구', '소공동', '회현동', '명동', '필동', '장충동', '광희동', '을지로동', '신당동', '다산동', '약수동', '청구동', '신당5동', '동화동', '황학동', '중림동']

  연령별 컬럼 unique 값: ['100세 이상', '65~69세', '70~74세', '75~79세', '80~84세', '85~89세', '90~94세', '95~99세', '합계']


In [7]:
# ── 등록인구(동별_연령별) 전처리 — 연도별 행정동별 전체인구 + 65세 이상 인구 ──

# ① 컬럼 식별
COL_DONG = df_raw_pop.columns[0]   # 동별
COL_AGE  = df_raw_pop.columns[1]   # 연령별

# 연간 컬럼: 'YYYY 년' 패턴 (분기 '/' 없음, 'Unnamed' 아님)
YEAR_COLS = {
    int(c.split()[0]): c
    for c in df_raw_pop.columns
    if c[:4].isdigit() and '/' not in c and 'Unnamed' not in c
}
print(f"연간 컬럼 매핑: {YEAR_COLS}")

# ② 서울시 25개 자치구명
SEOUL_GU_LIST = {
    '종로구', '중구', '용산구', '성동구', '광진구',
    '동대문구', '중랑구', '성북구', '강북구', '도봉구',
    '노원구', '은평구', '서대문구', '마포구', '양천구',
    '강서구', '구로구', '금천구', '영등포구', '동작구',
    '관악구', '서초구', '강남구', '송파구', '강동구',
}

# ③ gu forward-fill
REMOVE_DONG = SEOUL_GU_LIST | {'합계', 'nan', ''}
current_gu_val = None
gu_col = []
for val in df_raw_pop[COL_DONG]:
    if val in SEOUL_GU_LIST:
        current_gu_val = val
    gu_col.append(current_gu_val)
df_raw_pop['gu'] = gu_col

# ④ 동별 기본 필터: 합계·구 subtotal 제거
df_pop_base = df_raw_pop[~df_raw_pop[COL_DONG].isin(REMOVE_DONG)].copy()
df_pop_base = df_pop_base.rename(columns={COL_DONG: 'dong', COL_AGE: 'age'})
df_pop_base['dong'] = df_pop_base['dong'].str.strip()
df_pop_base['gu']   = df_pop_base['gu'].str.strip()

# ⑤ 65세 이상 연령 목록
ELDERLY_AGES = {
    '65~69세', '70~74세', '75~79세', '80~84세',
    '85~89세', '90~94세', '95~99세', '100세 이상',
}

# ⑥ 연도별 반복
records_dong_pop = []
for year, col_name in sorted(YEAR_COLS.items()):
    # 해당 연도 결측 제거 — NaN, 'nan', 빈 문자열 모두 제거
    val_str = df_pop_base[col_name].astype(str).str.strip()
    df_yr = df_pop_base[
        val_str.notna() &
        (val_str != 'nan') &
        (val_str != '') &
        (val_str != 'NaN')
    ].copy()
    df_yr['value'] = to_numeric_safe(df_yr[col_name])

    # total_population
    df_tot = (
        df_yr[df_yr['age'] == '합계'][['gu', 'dong', 'value']]
        .rename(columns={'value': 'total_population'})
    )

    # elderly_population
    df_eld = (
        df_yr[df_yr['age'].isin(ELDERLY_AGES)]
        .groupby(['gu', 'dong'], as_index=False)['value']
        .sum()
        .rename(columns={'value': 'elderly_population'})
    )

    df_yr_final = df_tot.merge(df_eld, on=['gu', 'dong'], how='inner')
    df_yr_final['year'] = year
    df_yr_final['dong_key'] = df_yr_final['gu'] + '_' + df_yr_final['dong']

    # 중복 확인
    dup = df_yr_final[df_yr_final.duplicated('dong_key', keep=False)]
    if len(dup) > 0:
        print(f"\n⚠️ {year}년 dong_key 중복 {len(dup)}건:")
        print(dup[['gu', 'dong', 'total_population']].to_string(index=False))
    else:
        print(f"{year}년: {len(df_yr_final)}개 행정동, 중복 없음 ✓")

    records_dong_pop.append(df_yr_final)

# ⑦ long format
df_dong_pop = pd.concat(records_dong_pop, ignore_index=True)
df_dong_pop = df_dong_pop[["year", "gu", "dong", "dong_key", "total_population", "elderly_population"]]

print(f"\n등록인구(동별) long format 완료: {len(df_dong_pop)}행")
print(f"연도별 행정동 수: {df_dong_pop.groupby('year')['dong_key'].nunique().to_dict()}")
print(f"\n샘플 (종로구_청운효자동, 3개 연도):")
print(df_dong_pop[df_dong_pop['dong_key'] == '종로구_청운효자동'].to_string(index=False))


연간 컬럼 매핑: {2019: '2019 년', 2020: '2020 년', 2021: '2021 년'}
2019년: 424개 행정동, 중복 없음 ✓
2020년: 425개 행정동, 중복 없음 ✓
2021년: 426개 행정동, 중복 없음 ✓

등록인구(동별) long format 완료: 1275행
연도별 행정동 수: {2019: 424, 2020: 425, 2021: 426}

샘플 (종로구_청운효자동, 3개 연도):
 year  gu  dong  dong_key  total_population  elderly_population
 2019 종로구 청운효자동 종로구_청운효자동             12981                2202
 2020 종로구 청운효자동 종로구_청운효자동             12633                2249
 2021 종로구 청운효자동 종로구_청운효자동             12177                2165


---
## 5. 장애유형별_동별 데이터 불러오기 및 전처리

### 파일 구조
- **헤더 행**: 4행 (행0=연도, 행1=합계, 행2=장애유형, 행3=계/남자/여자)  
- **지리 컬럼**: 동별(1)=고정값(합계), 동별(2)=구이름

### 연도별 소계/계 컬럼 위치
1년치 = 소계 + 15개 장애유형 = 16종 × 계/남/여 = **48열**

| 연도 | 소계/계 컬럼 인덱스 |
|------|-------------------|
| 2019 | 2 |
| 2020 | 50 (= 2 + 48) |
| 2021 | 98 (= 2 + 96) |

> ⚠️ **공간 단위 주의**: 이 파일도 파일명에 '동별'이 있으나 실제 데이터는  
> **자치구(구별) 수준**만 포함합니다.

In [8]:
# ── 장애인현황 파일 구조 확인 ──
print("=== 장애인현황 파일 ===\n읽는 중...")
df_raw_dis = read_csv_safe(FILE_DISABLED)

print("\n헤더 4행 (처음 8열):")
print(df_raw_dis.iloc[:4, :8].to_string(index=True))

print("\n데이터 시작 3행 (처음 4열):")
print(df_raw_dis.iloc[4:7, :4].to_string(index=True))

=== 장애인현황 파일 ===
읽는 중...
  → 인코딩 'utf-8-sig' 성공 (30행 × 146열)

헤더 4행 (처음 8열):
       0      1     2     3     4     5     6     7
0  동별(1)  동별(2)  2019  2019  2019  2019  2019  2019
1  동별(1)  동별(2)    합계    합계    합계    합계    합계    합계
2  동별(1)  동별(2)    소계    소계    소계    지체    지체    지체
3  동별(1)  동별(2)     계    남자    여자     계    남자    여자

데이터 시작 3행 (처음 4열):
    0    1       2       3
4  합계   소계  394843  228821
5  합계  종로구    6068    3515
6  합계   중구    5712    3282


In [9]:
# ── 장애인현황 전처리 (구별 장애인 총계) ──

# 1) 데이터 행 추출 (헤더 4행 건너뜀)
df_dis = df_raw_dis.iloc[4:].reset_index(drop=True).copy()
df_dis = df_dis.rename(columns={0: "동별1", 1: "gu"})

# 2) 연도별 소계/계(장애인 전체) 컬럼 위치
#    소계/계: 각 연도 첫 번째 데이터 컬럼
DISABLED_COL = {2019: 2, 2020: 50, 2021: 98}

# 3) 연도별 long format 변환
records_dis = []
for year, col_idx in DISABLED_COL.items():
    tmp = df_dis[["gu", col_idx]].copy()
    tmp.columns = ["gu", "disabled_population"]
    tmp["year"] = year
    records_dis.append(tmp)

df_gu_dis = pd.concat(records_dis, ignore_index=True)

# 4) 구 이름 공백 제거 및 요약행 제거
#    동별2='소계' 는 서울시 전체 합산행
df_gu_dis["gu"] = df_gu_dis["gu"].astype(str).str.strip()
df_gu_dis = df_gu_dis[
    ~df_gu_dis["gu"].isin(SUMMARY_KEYWORDS)
].reset_index(drop=True)

# 5) '-' 처리 및 숫자 변환
df_gu_dis["disabled_population"] = to_numeric_safe(df_gu_dis["disabled_population"])

print(f"장애인현황(구별) 처리 완료: {df_gu_dis.shape[0]}행")
print(f"연도별 구 수: {df_gu_dis.groupby('year')['gu'].count().to_dict()}")
print(f"\n샘플 (2021년):")
df_gu_dis[df_gu_dis["year"] == 2021].head(5)

장애인현황(구별) 처리 완료: 75행
연도별 구 수: {2019: 25, 2020: 25, 2021: 25}

샘플 (2021년):


,gu,disabled_population,year
50,종로구,5929,2021
51,중구,5634,2021
52,용산구,7686,2021
53,성동구,11284,2021
54,광진구,12253,2021


---
## 6. year + gu + dong 기준 데이터 병합

### 병합 전략

```
독거노인현황 (동별, 2019–2021)          ← 기준 데이터
       ↓ left join on [year, gu, dong]
등록인구 (동별, 2019–2021)              ← 행정동별 total + elderly
```

> **장애인 데이터 제외**: 구별 수준으로만 제공되어 행정동 단위 IVI 산출에 부적합  
> **행정동 개편 대응**: 병합 후 `total_population` 결측 행(양 데이터셋에 공통으로 없는 행정동)은 제거

In [10]:
# ── 데이터 병합 (2019–2021 전체) ──

# 독거노인(동별) + 등록인구(동별) → [year, gu, dong] 기준 병합
df_merged = df_elderly_all.merge(
    df_dong_pop[["year", "gu", "dong", "total_population", "elderly_population"]],
    on=["year", "gu", "dong"],
    how="left"
)

print(f"병합 결과: {df_merged.shape[0]}행 × {df_merged.shape[1]}열")

# 결측 진단 (행정동 개편 등으로 양 파일에 공통 존재하지 않는 행)
miss = df_merged["total_population"].isnull().sum()
if miss > 0:
    miss_rows = df_merged[df_merged["total_population"].isnull()]
    print(f"\n결측 {miss}건 → 행정동 개편으로 매칭 불가 → 제거 대상:")
    print(miss_rows[["year", "gu", "dong", "dong_key", "elderly_alone"]]
          .drop_duplicates("dong_key").to_string(index=False))

# 양 파일에 공통으로 매칭 가능한 행정동만 유지
df_merged = df_merged.dropna(subset=["total_population", "elderly_population"]).reset_index(drop=True)

print(f"\n결측 제거 후: {df_merged.shape[0]}행")
print(f"연도별 행 수: {df_merged.groupby('year').size().to_dict()}")
print(f"\n샘플 (종로구_청운효자동, 3개 연도):")
print(df_merged[df_merged["dong_key"] == "종로구_청운효자동"].to_string(index=False))


병합 결과: 1281행 × 7열

결측 6건 → 행정동 개편으로 매칭 불가 → 제거 대상:
 year  gu dong dong_key  elderly_alone
 2019 구로구   항동   구로구_항동              0
 2019 강동구 상일1동 강동구_상일1동              0
 2019 강동구 상일2동 강동구_상일2동              0
 2021 강동구  상일동  강동구_상일동              0

결측 제거 후: 1275행
연도별 행 수: {2019: 424, 2020: 425, 2021: 426}

샘플 (종로구_청운효자동, 3개 연도):
 gu  dong  elderly_alone  year  dong_key  total_population  elderly_population
종로구 청운효자동            476  2019 종로구_청운효자동           12981.0              2202.0
종로구 청운효자동            604  2020 종로구_청운효자동           12633.0              2249.0
종로구 청운효자동            514  2021 종로구_청운효자동           12177.0              2165.0


---
## 7. 병합 결과 검증

병합된 전체(2019–2021) 데이터의 구조와 인구 변별력을 확인합니다.  
`total_population`이 같은 구 내 행정동에서 **서로 다른 값**을 가져야 동별 데이터가 올바르게 적용된 것입니다.

In [11]:
# ── 병합 결과 검증 ──
df_all = df_merged.copy()

print("=== 연도별 행정동 수 ===")
print(df_all.groupby('year')['dong_key'].nunique())

print("\n=== 동별 변별력 확인 (2021년) ===")
df_2021_check = df_all[df_all['year'] == 2021]

for col in ["total_population", "elderly_population"]:
    nunique_per_gu = df_2021_check.groupby("gu")[col].nunique()
    flag = "✓ 동별 변별 정상" if nunique_per_gu.max() > 1 else "⚠️ 구 단위 반복!"
    print(f"\n[{col}]  min={nunique_per_gu.min()}, max={nunique_per_gu.max()}  {flag}")

print("\n=== dong_key 중복 확인 (2021) ===")
dup_check = df_2021_check[df_2021_check.duplicated('dong_key', keep=False)]
if len(dup_check) == 0:
    print("  중복 없음 ✓")
else:
    print(f"  ⚠️  중복 {len(dup_check)}건:")
    print(dup_check[['gu', 'dong', 'dong_key']].to_string(index=False))

print("\n=== 비율 범위 사전 확인 (2021) ===")
ep_ratio = df_2021_check['elderly_population'] / df_2021_check['total_population']
ea_ratio = df_2021_check['elderly_alone'] / df_2021_check['elderly_population']
print(f"  elderly_ratio  범위: {ep_ratio.agg(['min','max']).round(4).to_dict()}")
print(f"  elderly_alone_ratio 범위: {ea_ratio.agg(['min','max']).round(4).to_dict()}")


=== 연도별 행정동 수 ===

year
2019    424
2020    425
2021    426
Name: dong_key, dtype: int64

=== 동별 변별력 확인 (2021년) ===

[total_population]  min=10, max=27  ✓ 동별 변별 정상

[elderly_population]  min=10, max=26  ✓ 동별 변별 정상

=== dong_key 중복 확인 (2021) ===
  중복 없음 ✓

=== 비율 범위 사전 확인 (2021) ===
  elderly_ratio  범위: {'min': 0.0781, 'max': 0.3124}
  elderly_alone_ratio 범위: {'min': 0.0, 'max': 0.6358}


---
## 8. 비율 변수 생성

| 변수명 | 계산식 | 데이터 수준 |
|--------|--------|------------|
| `elderly_ratio` | elderly_population / total_population | 동별 / 동별 ✅ |
| `elderly_alone_ratio` | elderly_alone / elderly_population | 동별 / 동별 ✅ |

In [12]:
# ── 비율 변수 생성 ──

# ① elderly_ratio = 행정동 65세 이상 인구 / 행정동 전체 인구  (동별/동별 ✅)
df_all["elderly_ratio"] = (
    df_all["elderly_population"] / df_all["total_population"]
).replace([np.inf, -np.inf], np.nan)

# ② elderly_alone_ratio = 행정동 독거노인 수 / 행정동 65세 이상 인구  (동별/동별 ✅)
df_all["elderly_alone_ratio"] = (
    df_all["elderly_alone"] / df_all["elderly_population"]
).replace([np.inf, -np.inf], np.nan)

print("비율 변수 생성 완료")
print(f"\n[2021년 기초통계]")
df_2021 = df_all[df_all["year"] == 2021]
for col in ["elderly_ratio", "elderly_alone_ratio"]:
    s = df_2021[col]
    print(f"\n  [{col}]  min={s.min():.4f}  max={s.max():.4f}  "
          f"mean={s.mean():.4f}  결측={s.isnull().sum()}건")


비율 변수 생성 완료

[2021년 기초통계]

  [elderly_ratio]  min=0.0781  max=0.3124  mean=0.1686  결측=0건

  [elderly_alone_ratio]  min=0.0000  max=0.6358  mean=0.2458  결측=0건


---
## 9. 결측치 및 이상치 확인

결측치가 발생하면 **평균 대치보다 먼저 원인을 파악**합니다.

- **행정동명 불일치**: 독거노인 파일의 구 이름이 등록인구/장애인 파일의 구 이름과 다를 경우 left join 후 NaN 발생
- **요약행 미제거**: '소계', '합계' 등이 남아 있으면 집계가 왜곡됨
- **비율 > 1**: 분모(구별 인구)가 분자(동별 독거노인)보다 작으면 발생 → 구별 분모 사용의 한계

In [13]:
# ── 결측치 및 이상치 확인 ──
print("=== 결측치 현황 (전체 연도) ===")
miss = df_all.isnull().sum()
miss_show = miss[miss > 0]
if len(miss_show) == 0:
    print("  결측치 없음 ✓")
else:
    print(miss_show)

# ── 이상치 확인 (2021년 기준) ──
print("\n=== 이상치 확인 (2021년 기준) ===")
df_2021 = df_all[df_all["year"] == 2021]

ratio_high = df_2021[df_2021["elderly_ratio"] > 0.5]
print(f"  elderly_ratio > 0.5: {len(ratio_high)}건")

ratio_gt1 = df_2021[df_2021["elderly_alone_ratio"] > 1]
if len(ratio_gt1) > 0:
    print(f"\n  elderly_alone_ratio > 1: {len(ratio_gt1)}건")
    print(ratio_gt1[["gu", "dong", "elderly_alone", "elderly_population"]].to_string(index=False))
else:
    print(f"  elderly_alone_ratio > 1: 0건 ✓")

zero_ea = df_2021[df_2021["elderly_alone"] == 0]
print(f"\n  elderly_alone = 0인 동: {len(zero_ea)}건")
if len(zero_ea) > 0:
    print(zero_ea[["gu", "dong"]].to_string(index=False))


=== 결측치 현황 (전체 연도) ===
  결측치 없음 ✓

=== 이상치 확인 (2021년 기준) ===
  elderly_ratio > 0.5: 0건
  elderly_alone_ratio > 1: 0건 ✓

  elderly_alone = 0인 동: 1건
 gu dong
강동구  강일동


---
## 10. Min-Max 정규화

서로 다른 단위와 크기를 가진 변수들을 0–1 범위로 통일합니다.

$$\text{score} = \frac{x - \min(x)}{\max(x) - \min(x)}$$

- **0**: 가장 취약도가 낮은 동
- **1**: 가장 취약도가 높은 동
- 결측치(NaN)는 제외하고 계산하며, 결과도 NaN으로 유지합니다.

In [14]:
# ── Min-Max 정규화 (연도별 독립) ──
def minmax_normalize(series):
    min_val = series.min(skipna=True)
    max_val = series.max(skipna=True)
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series.apply(lambda x: 0.5 if pd.notna(x) else np.nan)
    return (series - min_val) / (max_val - min_val)


records_scored = []
for year, grp in df_all.groupby("year"):
    grp = grp.copy()
    grp["elderly_ratio_score"] = minmax_normalize(grp["elderly_ratio"])
    grp["elderly_alone_score"] = minmax_normalize(grp["elderly_alone_ratio"])
    records_scored.append(grp)

df_all = pd.concat(records_scored, ignore_index=True)

print("Min-Max 정규화 완료 (연도별 독립)")
print(f"\n[2021년 점수 분포]")
df_2021 = df_all[df_all["year"] == 2021]
for col in ["elderly_ratio_score", "elderly_alone_score"]:
    print(f"\n  [{col}]")
    print(df_2021[col].describe().round(4).to_string())


Min-Max 정규화 완료 (연도별 독립)

[2021년 점수 분포]

  [elderly_ratio_score]
count    426.0000
mean       0.3864
std        0.1505
min        0.0000
25%        0.2869
50%        0.3839
75%        0.4804
max        1.0000

  [elderly_alone_score]
count    426.0000
mean       0.3866
std        0.1184
min        0.0000
25%        0.3194
50%        0.3766
75%        0.4424
max        1.0000


---
## 11. IVI 산출

### 산출 공식 (2개 지표 동일가중 평균)

$$\text{IVI} = \frac{\text{elderly\_ratio\_score} + \text{elderly\_alone\_score}}{2}$$

| 구성 요소 | 계산식 | 데이터 수준 |
|----------|--------|------------|
| elderly_ratio_score | 동별 65세 이상 / 동별 전체 인구 → Min-Max | **동별 ✅** |
| elderly_alone_score | 동별 독거노인 / 동별 65세 이상 → Min-Max | **동별 ✅** |

> 장애인 데이터는 구별 수준만 제공되어 행정동 단위 IVI에서 제외합니다.

In [15]:
# ── IVI 산출 (2개 지표 동일가중 평균) ──
df_all["IVI"] = (df_all["elderly_ratio_score"] + df_all["elderly_alone_score"]) / 2

print("IVI 산출 완료")
print(f"공식: IVI = (elderly_ratio_score + elderly_alone_score) / 2")
print(f"\n[연도별 IVI 분포]")
print(df_all.groupby("year")["IVI"].describe().round(4))
print(f"\nIVI NaN: {df_all['IVI'].isnull().sum()}개")


IVI 산출 완료
공식: IVI = (elderly_ratio_score + elderly_alone_score) / 2

[연도별 IVI 분포]
      count    mean     std     min     25%     50%     75%     max
year                                                               
2019  424.0  0.2667  0.0803  0.0581  0.2139  0.2670  0.3103  0.6479
2020  425.0  0.3269  0.0956  0.0880  0.2643  0.3244  0.3769  0.7725
2021  426.0  0.3865  0.1155  0.1040  0.3067  0.3810  0.4504  0.8224

IVI NaN: 0개


---
## 12. IVI 순위 및 등급 생성

- **IVI_rank**: IVI가 높을수록 취약도가 높으므로, IVI 내림차순으로 순위를 부여합니다.  
  (rank 1 = 가장 취약한 행정동)
- **IVI_grade**: 5분위(20%씩)로 나누어 등급 부여  
  - A (상위 20%): 가장 취약  
  - E (하위 20%): 가장 덜 취약

In [16]:
# ── IVI 순위 및 등급 생성 (연도별 독립) ──

records_ranked = []
for year, grp in df_all.groupby("year"):
    grp = grp.copy()
    grp["IVI_rank"] = (
        grp["IVI"]
        .rank(ascending=False, method="min", na_option="bottom")
        .astype("Int64")
    )
    grp["IVI_grade"] = pd.qcut(
        grp["IVI"],
        q=5,
        labels=["E", "D", "C", "B", "A"],
        duplicates="drop"
    )
    records_ranked.append(grp)

df_all = pd.concat(records_ranked, ignore_index=True)

print("IVI 순위 및 등급 생성 완료 (연도별 독립)")
print(f"\n[연도별 등급 분포]")
print(df_all.groupby(["year", "IVI_grade"]).size().unstack(fill_value=0))

print(f"\n==== 2021년 IVI 상위 10개 행정동 (가장 취약) ====")
df_2021 = df_all[df_all["year"] == 2021]
top10 = df_2021.nsmallest(10, "IVI_rank")[
    ["IVI_rank", "gu", "dong",
     "elderly_ratio_score", "elderly_alone_score",
     "IVI", "IVI_grade"]
].reset_index(drop=True)
print(top10.to_string(index=False))


IVI 순위 및 등급 생성 완료 (연도별 독립)



[연도별 등급 분포]


IVI_grade   E   D   C   B   A
year                         
2019       85  85  84  85  85
2020       85  85  85  85  85
2021       86  85  85  85  85

==== 2021년 IVI 상위 10개 행정동 (가장 취약) ====
 IVI_rank   gu        dong  elderly_ratio_score  elderly_alone_score      IVI IVI_grade
        1  강남구         수서동             1.000000             0.644730 0.822365         A
        2   중구        을지로동             0.719260             0.913073 0.816167         A
        3  강북구         번3동             0.845551             0.774483 0.810017         A
        4  종로구 종로1.2.3.4가동             0.640018             0.975028 0.807523         A
        5  강서구        가양2동             0.917321             0.643003 0.780162         A
        6  용산구         남영동             0.508330             1.000000 0.754165         A
        7  강북구         번2동             0.738579             0.706835 0.722707         A
        8   중구         회현동             0.745543             0.662147 0.703845         A
        9 동대문구    

---
## 13. 최종 CSV 저장

최종 컬럼 순서로 정리하여 `data/processed/ivi_social_isolation_2021.csv`에 저장합니다.

- 인코딩: `utf-8-sig` (한글 Excel 호환)
- NaN 컬럼(`total_population`, `elderly_ratio`, `disabled_ratio`, `elderly_ratio_score`)은  
  데이터 한계를 명시하기 위해 컬럼 자체는 유지하되 NaN으로 저장합니다.

In [17]:
# ── 최종 컬럼 정리 및 CSV 저장 (2021년 기준) ──
df_2021 = df_all[df_all["year"] == 2021].copy().reset_index(drop=True)

OUTPUT_COLS = [
    "year",
    "gu",
    "dong",
    "dong_key",
    "total_population",
    "elderly_population",
    "elderly_alone",
    "elderly_ratio",
    "elderly_alone_ratio",
    "elderly_ratio_score",
    "elderly_alone_score",
    "IVI",
    "IVI_rank",
    "IVI_grade",
]

df_output = df_2021[OUTPUT_COLS].copy()

output_path = PROCESSED_DIR / "ivi_social_isolation_2021.csv"
df_output.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path}")
print(f"총 {len(df_output)}개 행정동, {len(df_output.columns)}개 컬럼")


저장 완료: c:\Tsum2026\T_SUM2026\data\processed\ivi_social_isolation_2021.csv
총 426개 행정동, 14개 컬럼


In [18]:
# ── 저장 파일 검증 ──
ivi = pd.read_csv(output_path, encoding="utf-8-sig")

print("=== 1. 연도 확인 ===")
print(ivi["year"].value_counts())

print("\n=== 2. dong_key 중복 확인 ===")
dup = ivi[ivi.duplicated("dong_key", keep=False)]
if len(dup) == 0:
    print("  중복 없음 ✓")
else:
    print(dup[["gu", "dong", "dong_key"]])

print("\n=== 3. 결측치 확인 ===")
miss = ivi.isnull().sum()
miss_show = miss[miss > 0]
if len(miss_show) == 0:
    print("  결측치 없음 ✓")
else:
    print(miss_show)

print("\n=== 4. 비율 범위 확인 ===")
for col in ["elderly_ratio", "elderly_alone_ratio"]:
    s = ivi[col]
    print(f"  [{col}]  min={s.min():.4f}  max={s.max():.4f}  >1={(s>1).sum()}건")

print("\n=== 5. IVI 재계산 검증 ===")
calc_ivi = (ivi["elderly_ratio_score"] + ivi["elderly_alone_score"]) / 2
max_diff = (calc_ivi - ivi["IVI"]).abs().max()
print(f"  최대 오차: {max_diff:.2e}  {'✓' if max_diff < 1e-9 else '⚠️'}")

print("\n=== 6. IVI 등급 분포 ===")
print(ivi["IVI_grade"].value_counts().sort_index())

print(f"\n행 수: {len(ivi)}, 열 수: {len(ivi.columns)}")
print("✓ 검증 완료")


=== 1. 연도 확인 ===
year
2021    426
Name: count, dtype: int64

=== 2. dong_key 중복 확인 ===
  중복 없음 ✓

=== 3. 결측치 확인 ===
  결측치 없음 ✓

=== 4. 비율 범위 확인 ===
  [elderly_ratio]  min=0.0781  max=0.3124  >1=0건
  [elderly_alone_ratio]  min=0.0000  max=0.6358  >1=0건

=== 5. IVI 재계산 검증 ===
  최대 오차: 1.11e-16  ✓

=== 6. IVI 등급 분포 ===
IVI_grade
A    85
B    85
C    85
D    85
E    86
Name: count, dtype: int64

행 수: 426, 열 수: 14
✓ 검증 완료


---
## 전처리 요약 및 한계

### 완료된 작업
| 단계 | 내용 | 결과 |
|------|------|------|
| 독거노인현황 | 동별 독거노인 총계 추출 (2019–2021) | ✅ |
| 등록인구(동별_연령별) | 동별 전체인구 + 65세이상 인구 (연간 합산) | ✅ |
| 병합 | year + gu + dong 기준 2개 데이터 결합 | ✅ |
| 비율 산출 | elderly_ratio, elderly_alone_ratio | ✅ |
| Min-Max 정규화 | 2개 점수 변수 (연도별 독립) | ✅ |
| IVI 산출 | 2개 지표 동일가중 평균 | ✅ |
| CSV 저장 | data/processed/ivi_social_isolation_2021.csv | ✅ |

### 장애인 데이터 제외 사유

- 장애인현황 파일은 파일명에 '동별'이 있으나 실제 데이터는 **자치구(구별) 수준**만 포함
- 구별 장애인 수 ÷ 동별 전체인구 = 소규모 동에서 ratio > 1 발생 (단위 불일치)
- 행정동 단위 IVI 산출에 부적합하여 제외

### 데이터 구조상 한계 (보고서 기재 권장)

1. **행정동 개편 대응**
   - 2019~2021 사이 행정동 수가 424→425→426으로 변동
   - 양 파일 모두에 매칭되지 않는 행정동은 `dropna`로 제거 (최종 2021년: 427개 기준)

2. **인구 기준 일치성**
   - 독거노인: 연간 데이터
   - 등록인구: 연간 합산(`YYYY 년` 컬럼) 사용
   - 소규모 동에서 연중 인구 변동이 있을 수 있음

3. **등록외국인 제외**
   - IVI는 한국인 정주 인구 기준으로 산출
   - 외국인 밀집 지역(구로구 가리봉동 등)에서 실제 고령 취약도가 과소 평가될 수 있음